# New experiment

This notebook lives on the lab's own disk and is gone when the lab scales down. Creating the experiment keeps it: the last saved version is copied to `experiments/<experiment_uid>/declaration.ipynb`, and to each run it declares as `declaring.ipynb`.

It tunes pretraining over a grid: `build` declares one run's whole chain, tokenizer to leg, from a point of `param_grid`, and `lab.sweep` returns the run of every point that validates.

In [ ]:
experiment_uid = "pretraining-sweep"  # one folder name, unused: a taken one is refused

In [4]:
import pandas as pd

import lab
from artifacts.core.artifact import Resources
from training.lr_schedule import WarmupCosine
from training.optimizers import AdamW
from training.leg import LoopConfig, TrainingParameters
from artifacts.mappeddataset import MappedDataSet
from artifacts.sources import SourceURL
from artifacts.stages.pretraining import Pretraining
from artifacts.stages.pretraining.evaluations import TokenLoss
from artifacts.stages.pretraining.samplers import RandomWindows
from artifacts.tokenizers.bpe import Tokenizer as BPETokenizer
from models import Model
from models.transformer import Transformer
from models.transformerFA import TransformerFA

## Model presets

Each row is a predefined shape. We use `tensor_dims_preset` ito pass the parameters into the `Model`

In [17]:
models = pd.DataFrame(
    [
        ("small", 768, 3072, 12, 12),
        ("medium", 1024, 4096, 24, 16),
        ("large", 1280, 5120, 36, 20),
        ("xl", 2560, 10240, 32, 32),
        ("10B", 4608, 12288, 50, 36),
    ],
    columns=["size", "d_model", "d_ff", "num_layers", "num_heads"],
).set_index("size")
tensor_dims_preset = lambda size : models.loc[size].to_dict()
models


,d_model,d_ff,num_layers,num_heads
size,,,,
small,768,3072,12,12
medium,1024,4096,24,16
large,1280,5120,36,20
xl,2560,10240,32,32
10B,4608,12288,50,36


## Sweep

`build` declares the run at one point of `param_grid`, everything it is built from included, and returns it; `lab.sweep` calls it at every point and returns the list. Each artifact validates as it is built, so a point any of them refuses, such as a `TransformerFA` without bf16 autocast, is printed with its reason and left out. Every key swept with more than one value is in the run's `run_id`. On an A10 (24 GB), the small preset at context 256 fits batch sizes up to 64; 128 does not.

In [ ]:
param_grid = {
    "vocab_size": [4096],  # each value trains its own tokenizer and tokenizes the sources again
    "model": [Transformer, TransformerFA],
    "size": ["small"],
    "context_length": [256],
    "batch_size": [16, 32, 64],
    "max_learning_rate": [3e-4],
    "total_steps" : [20000],
    "autocast": [None, "torch.bfloat16"],  # bf16 needs an L4, A10 or newer
    "compile": [None],  # or a torch.compile mode, "default" or "max-autotune"
    "gpu_type": ["A10"],
}


def run_id(conf) -> str:
    """Returns the experiment's uid and the value of every swept key, as one folder name."""
    label = lambda value: getattr(value, "__name__", str(value)).removeprefix("torch.")
    swept = [key for key, values in param_grid.items() if len(values) > 1]
    return "-".join([experiment_uid, *(f"{key}_{label(getattr(conf, key))}" for key in swept)])


def build(GRID) -> Pretraining:
    """Returns the run `conf` declares, with the tokenizer and dataset it trains on."""
    HF = "https://huggingface.co/datasets/roneneldan/TinyStories/resolve/main"
    train = SourceURL(name="tinystories-train", url=f"{HF}/TinyStoriesV2-GPT4-train.txt")
    valid = SourceURL(name="tinystories-valid", url=f"{HF}/TinyStoriesV2-GPT4-valid.txt")

    tokenizer = BPETokenizer(
        vocab_size=GRID.vocab_size,
        special_tokens=("<|endoftext|>",),
        sources=(valid,),
        allocated_resources=Resources(cpu=1),
    )
    dataset = MappedDataSet.from_sources(tokenizer=tokenizer, train_sources=(train,), valid_sources=(valid,),)
    pretraining = Pretraining(
        run_id=run_id(GRID),
        dataset=dataset,
        sampler=RandomWindows(),
        autocast=GRID.autocast,
        model=GRID.model(
            vocab_size=tokenizer.vocab_size,
            sequence_length=GRID.context_length,
            rope_theta=10000,
            tie_embeddings=False,
            dtype="torch.float32",
            **tensor_dims_preset(GRID.size),
        ),
        training_parameters=TrainingParameters(
            total_steps=GRID.total_steps,
            batch_size=GRID.batch_size,
            max_norm=1.0,
            lr_schedule=WarmupCosine(
                max_learning_rate=GRID.max_learning_rate,
                min_learning_rate=GRID.max_learning_rate / 10,
                warmup_iters=200,
                cosine_cycle_iters=GRID.total_steps,
            ),
            optimizer=AdamW(betas=(0.9, 0.95), weight_decay=0.1, eps=1e-8),
            seed=0,
        ),
        loop_config=LoopConfig(
            checkpoint_every=5000,
            evaluations=(TokenLoss(eval_every=200, batch_size=GRID.batch_size),),
            gpu_check_every=200,
            compile=GRID.compile,
        ),
        allocated_resources=Resources(gpu_type=GRID.gpu_type, gpu_count=1),
        tags={"experiment": [experiment_uid]},
    )
    return pretraining


runs = lab.sweep(build, param_grid)
for run in runs:
    print(run.artifact_path)

NameError: name 'valid' is not defined

## Create

Save the notebook first: the copy kept is the file on disk, and a notebook whose saved file lacks this cell is refused. `commit=False` previews, writing nothing.

In [ ]:
report = lab.create_experiment(experiment_uid, runs, verbose=True, commit=True)